# Day 16 – Embeddings & Vector Databases

Covers: what embeddings are, vector similarity, ChromaDB, FAISS, and semantic search.

#!pip install chromadb faiss-cpu sentence-transformers openai python-dotenv


In [1]:
import os
import numpy as np
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../../.env")

print("NumPy:", np.__version__)

NumPy: 1.26.4


---
## What Are Embeddings?

An **embedding** is a list of numbers (a vector) that represents the meaning of a piece of text.
Similar meanings → similar vectors. Different meanings → different vectors.

```
"dog"   → [0.2, 0.8, 0.1, ...]
"puppy" → [0.21, 0.79, 0.12, ...]   ← very similar
"car"   → [0.9, 0.1, 0.7, ...]      ← very different
```


In [2]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: small, fast, good quality — downloads ~80MB on first run
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "I love playing cricket.",
    "Cricket is my favourite sport.",
    "Machine learning is fascinating.",
    "Deep learning uses neural networks.",
    "I enjoy eating pizza.",
]

embeddings = model.encode(sentences)

print(f"Number of sentences : {len(embeddings)}")
print(f"Embedding dimension : {embeddings.shape[1]}")
print(f"First embedding (first 8 values): {embeddings[0][:8].round(4)}")

/Users/pradhnyesh/Documents/Linkific-training/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7659.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of sentences : 5
Embedding dimension : 384
First embedding (first 8 values): [ 0.0619  0.0313  0.0069 -0.0461  0.0039  0.0138  0.0534  0.0254]


---
## Vector Similarity

Two ways to measure how similar two vectors are:

- **Cosine similarity**: measures the angle between vectors. Range: -1 to 1. **1 = identical direction**.
- **Euclidean distance**: measures straight-line distance. **0 = identical**.

Cosine similarity is preferred for text because it ignores vector magnitude (sentence length).

In [3]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def euclidean_distance(a, b):
    return np.linalg.norm(a - b)

# Compare all pairs
print(f"{'Pair':<50} {'Cosine':>8} {'Euclidean':>10}")
print("-" * 70)
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        cos  = cosine_similarity(embeddings[i], embeddings[j])
        euc  = euclidean_distance(embeddings[i], embeddings[j])
        pair = f"{sentences[i][:22]} | {sentences[j][:22]}"
        print(f"{pair:<50} {cos:>8.4f} {euc:>10.4f}")

Pair                                                 Cosine  Euclidean
----------------------------------------------------------------------
I love playing cricket | Cricket is my favourit      0.8335     0.5770
I love playing cricket | Machine learning is fa      0.1628     1.2940
I love playing cricket | Deep learning uses neu      0.0583     1.3724
I love playing cricket | I enjoy eating pizza.       0.3703     1.1222
Cricket is my favourit | Machine learning is fa      0.2329     1.2387
Cricket is my favourit | Deep learning uses neu      0.0825     1.3546
Cricket is my favourit | I enjoy eating pizza.       0.2826     1.1978
Machine learning is fa | Deep learning uses neu      0.5381     0.9611
Machine learning is fa | I enjoy eating pizza.       0.1663     1.2913
Deep learning uses neu | I enjoy eating pizza.       0.0241     1.3971


In [4]:
# Semantic search: find the most similar sentence to a query
def find_most_similar(query, corpus_sentences, corpus_embeddings):
    query_emb = model.encode([query])[0]
    scores = [cosine_similarity(query_emb, e) for e in corpus_embeddings]
    best_idx = int(np.argmax(scores))
    return corpus_sentences[best_idx], scores[best_idx]

query = "I like watching football matches"
match, score = find_most_similar(query, sentences, embeddings)
print(f"Query : {query}")
print(f"Match : {match}")
print(f"Score : {score:.4f}")

Query : I like watching football matches
Match : Cricket is my favourite sport.
Score : 0.5740


---
## ChromaDB – Vector Database

ChromaDB stores embeddings and lets you search them by similarity.
It runs in-memory or persists to disk. No server needed.

In [ ]:
import chromadb

# In-memory client (data lost on restart)
client = chromadb.Client()

# Create a collection (like a table)
collection = client.create_collection(name="ml_facts")

# Documents to store
docs = [
    "Overfitting happens when a model memorises training data and performs poorly on new data.",
    "Gradient descent is an optimisation algorithm that minimises the loss function.",
    "A neural network has input, hidden, and output layers.",
    "Regularisation techniques like L1 and L2 help prevent overfitting.",
    "Cross-validation splits data into folds to evaluate model performance reliably.",
    "Feature engineering creates new input variables to improve model accuracy.",
    "The learning rate controls how large each gradient descent step is.",
    "Unsupervised learning finds patterns in data without using labels.",
]

# Add documents (ChromaDB generates embeddings using its built-in model)
collection.add(
    documents=docs,
    ids=[f"doc_{i}" for i in range(len(docs))]
)

print(f"Collection '{collection.name}' has {collection.count()} documents.")

/Users/pradhnyesh/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:41<00:00, 2.00MiB/s]


Collection 'ml_facts' has 8 documents.


In [ ]:
# Query the collection
results = collection.query(
    query_texts=["how to fix a model that memorises training data"],
    n_results=3
)

print("Top 3 results:")
for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0]), 1):
    print(f"  {i}. [{dist:.4f}] {doc}")

Top 3 results:
  1. [0.5591] Overfitting happens when a model memorises training data and performs poorly on new data.
  2. [1.3081] The learning rate controls how large each gradient descent step is.
  3. [1.3112] Unsupervised learning finds patterns in data without using labels.


In [ ]:
# Persistent ChromaDB (saves to disk)
persist_client = chromadb.PersistentClient(path="./chroma_db")

# get_or_create avoids error if collection already exists
persist_col = persist_client.get_or_create_collection(name="persistent_facts")

persist_col.add(
    documents=docs,
    ids=[f"pdoc_{i}" for i in range(len(docs))]
)

print(f"Persistent collection has {persist_col.count()} documents.")
print("Data saved to ./chroma_db/ — survives Python restarts.")

Persistent collection has 8 documents.
Data saved to ./chroma_db/ — survives Python restarts.


---
## FAISS – Fast Similarity Search

FAISS (by Meta) is a library for extremely fast nearest-neighbour search.
Unlike ChromaDB, FAISS only stores vectors — you manage the text yourself.
Best for very large datasets (millions of vectors).

In [ ]:
import faiss

# Build embeddings for our documents
doc_embeddings = model.encode(docs).astype("float32")
dimension = doc_embeddings.shape[1]  # 384 for MiniLM

# Create a flat (exact) index using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)
print(f"FAISS index has {index.ntotal} vectors of dimension {dimension}")

# Search
query = "techniques to prevent overfitting"
query_vec = model.encode([query]).astype("float32")

distances, indices = index.search(query_vec, k=3)  # top 3

print(f"\nQuery: '{query}'")
print("Top 3 matches:")
for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
    print(f"  {rank}. [dist={dist:.4f}] {docs[idx]}")

FAISS index has 8 vectors of dimension 384

Query: 'techniques to prevent overfitting'
Top 3 matches:
  1. [dist=0.5752] Regularisation techniques like L1 and L2 help prevent overfitting.
  2. [dist=0.6601] Overfitting happens when a model memorises training data and performs poorly on new data.
  3. [dist=1.3398] Cross-validation splits data into folds to evaluate model performance reliably.


---
## ChromaDB vs FAISS

| Feature | ChromaDB | FAISS |
|---|---|---|
| Stores text | Yes | No (vectors only) |
| Built-in embeddings | Yes | No |
| Persistence | Yes (PersistentClient) | Manual (save/load index) |
| Speed at scale | Good | Excellent |
| Best for | Small-medium apps, RAG | Millions of vectors |
| Setup complexity | Low | Medium |

**For RAG projects → use ChromaDB.** FAISS is better when you have millions of vectors and need raw speed.